# Logistic Regression on Medical Insurance Dataset

**Objective:** Predict whether a person is a **smoker** (Yes/No) based on features like age, sex, BMI, number of children, region, and medical charges.

**Dataset:** `medical.csv`

**Target Variable:** `smoker` (binary: yes / no)

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## 2. Load the Dataset

In [ ]:
df = pd.read_csv(r'../../medical.csv')
print(f'Dataset shape: {df.shape}')
df.head(10)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Basic info about the dataset
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistical Summary ===')
df.describe()

In [ ]:
# Check for missing values
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

In [ ]:
# Check for duplicate rows
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Data types and unique values
print('=== Data Types ===')
print(df.dtypes)
print('\n=== Unique Values per Column ===')
for col in df.columns:
    print(f'{col}: {df[col].nunique()} unique values')

In [ ]:
# Target variable distribution
print('=== Smoker Distribution ===')
print(df['smoker'].value_counts())
print(f'\nPercentage:\n{df["smoker"].value_counts(normalize=True) * 100}')

plt.figure(figsize=(6, 4))
sns.countplot(x='smoker', data=df, palette='Set2')
plt.title('Distribution of Smoker (Target Variable)')
plt.xlabel('Smoker')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of numerical features
numerical_cols = ['age', 'bmi', 'children', 'charges']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(numerical_cols):
    ax = axes[i // 2, i % 2]
    sns.histplot(df[col], kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: Numerical features vs Smoker
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for i, col in enumerate(numerical_cols):
    ax = axes[i // 2, i % 2]
    sns.boxplot(x='smoker', y=col, data=df, ax=ax, palette='Set2')
    ax.set_title(f'{col} vs Smoker')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features vs Smoker
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x='sex', hue='smoker', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Sex vs Smoker')

sns.countplot(x='region', hue='smoker', data=df, ax=axes[1], palette='Set2')
axes[1].set_title('Region vs Smoker')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap (after encoding)
df_encoded_temp = df.copy()
df_encoded_temp['sex'] = LabelEncoder().fit_transform(df_encoded_temp['sex'])
df_encoded_temp['smoker'] = LabelEncoder().fit_transform(df_encoded_temp['smoker'])
df_encoded_temp['region'] = LabelEncoder().fit_transform(df_encoded_temp['region'])

plt.figure(figsize=(8, 6))
sns.heatmap(df_encoded_temp.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

## 4. Data Preprocessing

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Encode categorical variables using Label Encoding
le_sex = LabelEncoder()
le_smoker = LabelEncoder()
le_region = LabelEncoder()

df_processed['sex'] = le_sex.fit_transform(df_processed['sex'])
df_processed['smoker'] = le_smoker.fit_transform(df_processed['smoker'])
df_processed['region'] = le_region.fit_transform(df_processed['region'])

print('Encoding Mappings:')
print(f'  sex     : {dict(zip(le_sex.classes_, le_sex.transform(le_sex.classes_)))}')
print(f'  smoker  : {dict(zip(le_smoker.classes_, le_smoker.transform(le_smoker.classes_)))}')
print(f'  region  : {dict(zip(le_region.classes_, le_region.transform(le_region.classes_)))}')

print('\nProcessed Dataset:')
df_processed.head()

## 5. Define Features (X) and Target (y)

In [ ]:
# Features and Target
X = df_processed.drop('smoker', axis=1)
y = df_processed['smoker']

print(f'Features shape: {X.shape}')
print(f'Target shape  : {y.shape}')
print(f'\nFeature columns: {list(X.columns)}')
print(f'\nTarget distribution:\n{y.value_counts()}')

## 6. Train-Test Split

In [ ]:
# Split data into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set size: {X_train.shape[0]}')
print(f'Testing set size : {X_test.shape[0]}')
print(f'\nTraining target distribution:\n{y_train.value_counts()}')
print(f'\nTesting target distribution:\n{y_test.value_counts()}')

## 7. Feature Scaling

In [ ]:
# Standardize features for better convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Feature scaling completed!')
print(f'X_train_scaled shape: {X_train_scaled.shape}')
print(f'X_test_scaled shape : {X_test_scaled.shape}')

## 8. Train Logistic Regression Model

In [ ]:
# Create and train the Logistic Regression model
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

print('Logistic Regression model trained successfully!')
print(f'\nModel Coefficients:')
coeff_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0]
}).sort_values(by='Coefficient', ascending=False)
print(coeff_df.to_string(index=False))
print(f'\nIntercept: {log_reg.intercept_[0]:.4f}')

## 9. Predictions

In [ ]:
# Make predictions on the test set
y_pred = log_reg.predict(X_test_scaled)
y_pred_proba = log_reg.predict_proba(X_test_scaled)[:, 1]

# Display first 10 predictions vs actual
results = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Probability (Smoker)': y_pred_proba[:10].round(4)
})
results['Actual Label'] = results['Actual'].map({0: 'No', 1: 'Yes'})
results['Predicted Label'] = results['Predicted'].map({0: 'No', 1: 'Yes'})
print('First 10 Predictions vs Actual:')
results

## 10. Model Evaluation

In [ ]:
# Accuracy Score
train_accuracy = log_reg.score(X_train_scaled, y_train)
test_accuracy = accuracy_score(y_test, y_pred)

print(f'Training Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)')
print(f'Testing Accuracy : {test_accuracy:.4f} ({test_accuracy*100:.2f}%)')

In [ ]:
# Classification Report
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['Non-Smoker (0)', 'Smoker (1)']))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print('=== Confusion Matrix ===')
print(cm)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Non-Smoker', 'Smoker'],
            yticklabels=['Non-Smoker', 'Smoker'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()

# Extract values
tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negatives  (TN): {tn}')
print(f'False Positives (FP): {fp}')
print(f'False Negatives (FN): {fn}')
print(f'True Positives  (TP): {tp}')

In [ ]:
# ROC Curve and AUC Score
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
auc_score = roc_auc_score(y_test, y_pred_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend(loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'AUC Score: {auc_score:.4f}')

## 11. Feature Importance Visualization

In [ ]:
# Feature importance based on model coefficients
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': log_reg.coef_[0],
    'Abs_Coefficient': np.abs(log_reg.coef_[0])
}).sort_values(by='Abs_Coefficient', ascending=True)

plt.figure(figsize=(8, 5))
colors = ['green' if c > 0 else 'red' for c in feature_importance['Coefficient']]
plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.title('Feature Importance (Logistic Regression Coefficients)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 12. Summary & Conclusion

In [ ]:
print('=' * 60)
print('       LOGISTIC REGRESSION MODEL SUMMARY')
print('=' * 60)
print(f'  Dataset              : medical.csv')
print(f'  Total Samples        : {df.shape[0]}')
print(f'  Features Used        : {list(X.columns)}')
print(f'  Target Variable      : smoker (0 = No, 1 = Yes)')
print(f'  Train / Test Split   : 80% / 20%')
print(f'  Training Accuracy    : {train_accuracy*100:.2f}%')
print(f'  Testing Accuracy     : {test_accuracy*100:.2f}%')
print(f'  AUC Score            : {auc_score:.4f}')
print(f'  True Positives       : {tp}')
print(f'  True Negatives       : {tn}')
print(f'  False Positives      : {fp}')
print(f'  False Negatives      : {fn}')
print('=' * 60)
print('\nConclusion:')
print('The Logistic Regression model was trained to predict smoker')
print('status from the medical insurance dataset. The model uses')
print('age, sex, BMI, children, region, and charges as features.')
print(f'It achieved a testing accuracy of {test_accuracy*100:.2f}% and')
print(f'an AUC score of {auc_score:.4f}, indicating good')
print('discriminative ability between smokers and non-smokers.')